# Qwen follow-up analyses
Error by interoceptivity tertile (fig 8), calibration / entropy (fig 9), error distribution histograms (fig 10), and top over- and under-rated words (table 3).

## Load data and Qwen probability distributions

In [ ]:
import sys, json
sys.path.insert(0, r"C:\\Users\\PauloPC2019\\Desktop\\Analysis scriptie")
from data_loading import load_data
import numpy as np

df = load_data()

qwen_path = r"C:\\Users\\PauloPC2019\\Desktop\\Analysis scriptie\\prediction_qwen_temp0_full_data_set.jsonl"
with open(qwen_path) as f:
    q_raw = __import__("pandas").DataFrame(json.loads(f.read()))[["WORD", "probs"]]
df = df.merge(q_raw, on="WORD", how="left")

def entropy(p_dict):
    p = np.array([p_dict.get(str(i), 0.0) for i in range(6)], dtype=float)
    p = p[p > 0]
    if len(p) == 0: return 0.0
    return float(-(p * np.log2(p)).sum())

df["qwen_entropy"]      = df["probs"].apply(entropy)
df["qwen_error_signed"] = df["qwen_expected"] - df["human"]
df["qwen_error_abs"]    = df["qwen_error_signed"].abs()
df.head()


## Fig 8 - error by human-rating tertile

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

bins = pd.qcut(df["human"], q=3, labels=["Low\n(human <= 0.5)", "Mid\n(0.5 - 1.4)", "High\n(human > 1.4)"])
bin_lbl = ["Low\n(human <= 0.5)", "Mid\n(0.5 - 1.4)", "High\n(human > 1.4)"]
signed = [df.loc[bins == b, "qwen_error_signed"].mean() for b in bin_lbl]
abs_e  = [df.loc[bins == b, "qwen_error_abs"   ].mean() for b in bin_lbl]
ns     = [int((bins == b).sum()) for b in bin_lbl]

fig, ax = plt.subplots(figsize=(8, 5.5), dpi=200)
x = np.arange(3); w = 0.35
ax.bar(x - w/2, signed, w, label="Mean signed error (expected - human)",
       color="#9467bd", edgecolor="black", linewidth=0.6)
ax.bar(x + w/2, abs_e,  w, label="Mean absolute error",
       color="#cdb4db", edgecolor="black", linewidth=0.6)
for xi, v in zip(x - w/2, signed):
    ax.text(xi, v + 0.02, f"{v:+.2f}", ha="center", va="bottom", fontsize=10)
for xi, v in zip(x + w/2, abs_e):
    ax.text(xi, v + 0.02, f"{v:.2f}",  ha="center", va="bottom", fontsize=10)
ax.axhline(0, color="black", linewidth=0.6)
ax.set_xticks(x); ax.set_xticklabels([f"{b}\nn = {n:,}" for b, n in zip(bin_lbl, ns)], fontsize=10)
ax.set_ylabel("Qwen error (0-5 scale)", fontsize=11)
ax.set_title("Qwen prediction error by human-rating tertile", fontsize=12, fontweight="bold")
ax.legend(frameon=True, fontsize=10, loc="upper right")
ax.grid(True, axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("fig8_qwen_error_by_bin.png", dpi=200, bbox_inches="tight")
plt.show()


## Fig 9 - calibration (entropy vs. error)

In [ ]:
df["entropy_q"] = pd.qcut(df["qwen_entropy"], q=4,
                          labels=["Q1\n(lowest)", "Q2", "Q3", "Q4\n(highest)"])
ent_summary = df.groupby("entropy_q", observed=True).agg(
    abs_error=("qwen_error_abs",    "mean"),
    signed_error=("qwen_error_signed","mean"),
    n=("WORD", "count"),
    entropy_mean=("qwen_entropy", "mean"),
)

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=200)
x = np.arange(len(ent_summary)); w = 0.35
b1 = ax.bar(x - w/2, ent_summary["signed_error"], w,
            label="Mean signed error", color="#9467bd", edgecolor="black", linewidth=0.6)
b2 = ax.bar(x + w/2, ent_summary["abs_error"],    w,
            label="Mean absolute error", color="#cdb4db", edgecolor="black", linewidth=0.6)
for bar, v in zip(b1, ent_summary["signed_error"]):
    ax.text(bar.get_x()+bar.get_width()/2, v + 0.015, f"{v:+.2f}",
            ha="center", va="bottom", fontsize=10)
for bar, v in zip(b2, ent_summary["abs_error"]):
    ax.text(bar.get_x()+bar.get_width()/2, v + 0.015, f"{v:.2f}",
            ha="center", va="bottom", fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels([f"Q{i+1}\nentropy ~ {e:.2f}\nn = {n:,}"
                    for i, (e, n) in enumerate(zip(ent_summary["entropy_mean"], ent_summary["n"]))],
                   fontsize=9)
ax.set_ylabel("Qwen error (0-5 scale)", fontsize=11)
ax.set_xlabel("Entropy quartile (bits)", fontsize=11)
ax.set_title("Qwen prediction error by uncertainty (entropy of probability distribution)",
             fontsize=12, fontweight="bold")
ax.axhline(0, color="black", linewidth=0.6)
ax.grid(True, axis="y", linestyle="--", alpha=0.4)
ax.legend(frameon=True, fontsize=10, loc="upper right")
plt.tight_layout()
plt.savefig("fig9_qwen_calibration.png", dpi=200, bbox_inches="tight")
plt.show()


## Fig 10 - error distribution histograms

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=200)

ax = axes[0]
ax.hist(df["qwen_error_signed"], bins=60, color="#9467bd", edgecolor="white", linewidth=0.3)
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.axvline(df["qwen_error_signed"].mean(), color="red", linestyle="-", linewidth=1.2,
           label=f"Mean = {df['qwen_error_signed'].mean():+.2f}")
ax.set_xlabel("Signed error (Qwen - human)", fontsize=11)
ax.set_ylabel("Number of words", fontsize=11)
ax.set_title("Signed error distribution", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, linestyle="--", alpha=0.4)

ax = axes[1]
ax.hist(df["qwen_error_abs"], bins=60, color="#cdb4db", edgecolor="white", linewidth=0.3)
ax.axvline(df["qwen_error_abs"].mean(), color="red", linestyle="-", linewidth=1.2,
           label=f"Mean = {df['qwen_error_abs'].mean():.2f}")
within_half = (df["qwen_error_abs"] <= 0.5).mean() * 100
within_one  = (df["qwen_error_abs"] <= 1.0).mean() * 100
ax.set_xlabel("Absolute error |Qwen - human|", fontsize=11)
ax.set_ylabel("Number of words", fontsize=11)
ax.set_title("Absolute error distribution", fontsize=12, fontweight="bold")
ax.text(0.97, 0.85, f"{within_half:.0f}% within 0.5\n{within_one:.0f}% within 1.0",
        transform=ax.transAxes, fontsize=10, ha="right", va="top",
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="gray"))
ax.legend(fontsize=10)
ax.grid(True, linestyle="--", alpha=0.4)

fig.suptitle("Distribution of Qwen prediction errors against human ratings",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig10_qwen_error_distribution.png", dpi=200, bbox_inches="tight")
plt.show()


## Top 10 most over-rated and under-rated words

In [ ]:
cols = ["WORD", "human", "qwen_expected", "qwen_error_signed", "qwen_error_abs", "qwen_entropy"]
over  = df.sort_values("qwen_error_signed", ascending=False).head(10)[cols]
under = df.sort_values("qwen_error_signed", ascending=True ).head(10)[cols]

print("=== Most over-rated by Qwen ===")
print(over.to_string(index=False))
print("\n=== Most under-rated by Qwen ===")
print(under.to_string(index=False))
